# FinSight — Financial News Risk Intelligence System
## WMG9B7 Individual Assessment | Sourabha K Kallapur

---

## Problem Statement

Financial institutions process thousands of news articles daily to assess market risk and make time-sensitive investment decisions. Manual review is prohibitively slow and inconsistent, while keyword-based systems miss nuanced financial signals. FinSight automates this pipeline by classifying financial news by topic, scoring urgency from tabular metadata, and generating structured risk briefs using a fine-tuned DistilBERT model. The system detects market regime shifts through statistical drift monitoring, enabling proactive risk management at scale.

## Setup Instructions

Run the following before executing any other cell:

```bash
!pip install -e ..
```

All cells must be run **in order**.  
Dataset loads automatically via HuggingFace `datasets`.

## Expected Runtime

| Environment       | Duration    |
|-------------------|-------------|
| Colab GPU (T4)    | ~45 minutes |
| CPU only          | ~3 hours    |

## Model Artefacts

- If `artefacts/distilbert_finsight.pt` **exists**, the notebook loads it and skips training.  
- If it does **not** exist, the notebook trains DistilBERT from scratch (3 epochs, ~14k samples).

## Problem Framing and Motivation

### The Problem — Why Financial News Classification Matters

Financial institutions must process thousands of news articles daily to assess market risk, yet manual review is prohibitively slow for real-time decision-making. Misclassifying a market-moving event — such as a central bank policy change embedded in diplomatic language — can result in significant financial losses within minutes of publication. Automated, high-accuracy classification enables real-time alerting, portfolio rebalancing triggers, and compliance logging. The accuracy and latency of such a system directly translate to competitive advantage and quantifiable risk mitigation.

### Why Deep Learning Over Classical ML

Traditional TF-IDF approaches treat text as a bag of words, discarding word order and contextual relationships. Consider: *“Fed raises rates unexpectedly, markets fall”* and *“Markets raise the Fed’s unexpected rates fall”* — identical TF-IDF vectors, very different meanings. DistilBERT (Sanh et al., 2019), a knowledge-distilled variant of BERT (Devlin et al., 2019), learns contextual embeddings where the same token receives different representations depending on its neighbours. The empirical comparison in this notebook demonstrates that contextual understanding yields substantially higher classification accuracy, particularly for ambiguous financial headlines.

### System Overview

FinSight is a 5-layer production pipeline:

| Layer | Component | Purpose |
|-------|-----------|----------|
| 1 | **Ingestion** | Pydantic schema validation and metadata extraction |
| 2 | **Preprocessing** | HTML stripping, URL removal, Unicode normalisation |
| 3 | **Classification** | Dual-model: TF-IDF + LogReg (fast) and DistilBERT (accurate) |
| 4 | **Risk Scoring** | Urgency assessment from 7 tabular metadata features |
| 5 | **Monitoring** | PSI / KS / Chi-Square statistical drift detection |

In [ ]:
# Install package in editable mode so src/ imports work in Colab
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".."],
    capture_output=True,
    text=True,
)
if result.returncode == 0:
    print("Package installed successfully.")
else:
    print(result.stdout[-1000:])
    print(result.stderr[-1000:])
    raise RuntimeError("Package installation failed.")

# Ensure project root is on sys.path
from pathlib import Path

_project_root = str(Path("..").resolve())
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)

import json
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sklearn.metrics import confusion_matrix, f1_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

LABEL_NAMES = ["World", "Sports", "Business", "Sci/Tech"]
LABEL_TO_INT = {"World": 0, "Sports": 1, "Business": 2, "Sci/Tech": 3}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Python {sys.version.split()[0]}")
print(f"PyTorch {torch.__version__}")